# MTSD 批量切片 + 缩放导出（复用 slicer）

将 `mtsd` 的 `train_full / train_partial / val` 下的 images+labels 批量裁切（1280, overlap 0.2）并生成 letterbox 缩放图，输出到 `/Volumes/Weixian Fu/mtsd/resize1280/<split>/{images,labels}`。切片命名 `<stem>_<idx>`，缩放命名 `<stem>_F`。tqdm 显示进度，可多线程。


In [1]:
import sys
from pathlib import Path
from tqdm import tqdm
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed

# 项目根加入路径，确保能导入 src
sys.path.append("/Users/weixianfu/Documents/Projects/road-sign-eu-mvp")

from src.io.slicer import save_slices_and_resized


In [2]:
# 配置
SRC_ROOT = Path("/Users/weixianfu/Documents/Datas/mtsd")
SPLITS = ["train_full", "train_partial", "val"]
OUT_ROOT = Path("/Users/weixianfu/Documents/Datas/mtsd-resized")
SLICE_SIZE = 1280
OVERLAP = 0.2
RESIZE = 1280
WORKERS = 8  # >1 启用多线程；设为1串行



In [3]:
# 批量转换（先清空每个 split 的输出，再累积写入；每图调用 slicer 封装）
for split in SPLITS:
    src_images = SRC_ROOT / split / "images"
    if not src_images.exists():
        print(f"skip {split}, images dir missing: {src_images}")
        continue

    out_split = OUT_ROOT / split
    # 清空目标 split
    if out_split.exists():
        shutil.rmtree(out_split)
    (out_split / "images").mkdir(parents=True, exist_ok=True)
    (out_split / "labels").mkdir(parents=True, exist_ok=True)

    img_files = sorted([p for p in src_images.glob("*") if p.is_file()])
    print(f"Processing {split}: {len(img_files)} images")

    def worker(p: Path):
        save_slices_and_resized(
            str(p),
            str(out_split),
            slice_size=SLICE_SIZE,
            overlap=OVERLAP,
            resize_size=RESIZE,
            clear_output=False,  # 仅分组开始时清空一次
        )

    if WORKERS <= 1:
        for p in tqdm(img_files, desc=f"{split}", unit="img"):
            worker(p)
    else:
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futures = [ex.submit(worker, p) for p in img_files]
            for _ in tqdm(as_completed(futures), total=len(futures), desc=f"{split}", unit="img"):
                pass

    print(f"Done {split} -> {out_split}")



Processing train_full: 36589 images


train_full: 100%|██████████| 36589/36589 [06:52<00:00, 88.74img/s] 


Done train_full -> /Users/weixianfu/Documents/Datas/mtsd-resized/train_full
Processing train_partial: 53377 images


train_partial: 100%|██████████| 53377/53377 [10:50<00:00, 82.01img/s] 


Done train_partial -> /Users/weixianfu/Documents/Datas/mtsd-resized/train_partial
Processing val: 5320 images


val: 100%|██████████| 5320/5320 [00:57<00:00, 92.08img/s] 

Done val -> /Users/weixianfu/Documents/Datas/mtsd-resized/val
